In [1]:
import json
import os

import pandas as pd

# --- Local (with a .env file) ---
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]


# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What is land breeze?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage)

A land breeze is a local wind that blows from the land to the sea, typically at night. It occurs when the land cools faster than the sea after sunset, causing the air over the land to become cooler and denser than the air over the sea.

As the land cools, the air above it contracts and becomes heavier, creating a pressure gradient that pushes the air from the land towards the sea. This movement of air from the land to the sea is called a land breeze.

Land breezes are usually weaker than sea breezes, which blow from the sea to the land during the day. They are also typically more localized and may not extend as far out to sea as sea breezes.

Land breezes can have several effects, including:

1. Cooling the land: By blowing from the land to the sea, land breezes can help to cool the land by removing heat from the surface.
2. Reducing fog: Land breezes can help to reduce fog by blowing it out to sea.
3. Affecting marine life: Land breezes can affect the distribution of marine life, such

## Student Reasoning — Anatomy of a Call

1. Difference between system and user

System: Provides the AI with instructions about how it should behave, respond, or what role it should perform.
Example: “Act as a helpful mathematics tutor.”
User: Contains the specific question, task, or instruction that the AI needs to answer.
Example: “Explain eigenvalues and eigenvectors in simple terms.”

2. What is a token?

A token is a small unit of text that a large language model (LLM) reads and processes. It can be a complete word, part of a word, or sometimes a character.

Why do API providers charge based on tokens instead of requests?

API providers use tokens for billing because each request can contain and produce different amounts of text. Counting tokens gives a more accurate way to measure how much information the model processes and generates.

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0
# and 5 times at temperature=1.2.

# TODO: Print all 10 answers, grouped by temperature.
question = "What is reverse osmosis?"

print("--- Temperature 0.0 ---")
for i in range(5):
  answer = ask_llm(question, temperature=0.0)
  print(f"{i + 1}. {answer.choices[0].message.content}")

print("\n--- Temperature 1.2 ---")
for i in range(5):
  answer = ask_llm(question, temperature=1.2)
  print(f"{i + 1}. {answer.choices[0].message.content}")

--- Temperature 0.0 ---
1. Reverse osmosis (RO) is a water purification process that uses a semi-permeable membrane to remove impurities and contaminants from water. The process involves applying pressure to force the water through the membrane, which has tiny pores that allow water molecules to pass through while blocking larger particles and impurities.

Here's a step-by-step explanation of the reverse osmosis process:

1. **Pre-treatment**: The water is pre-treated to remove larger particles and debris that could damage the membrane.
2. **Pressurization**: The pre-treated water is pressurized to force it through the semi-permeable membrane.
3. **Filtration**: The pressurized water is forced through the membrane, which has pores that are typically 0.0001 microns in size. This size is small enough to block most impurities, including:
	* Dissolved solids (e.g., salt, minerals)
	* Bacteria
	* Viruses
	* Heavy metals
	* Pesticides
	* Herbicides
4. **Separation**: The water molecules pass

## Student Reasoning — Temperature
i Temperature = 0.0: The responses were highly consistent and had very similar wording, structure, and ideas when explaining the main concepts of banking systems.
Temperature = 1.2: The responses showed greater variation. The model used different wording, examples, and response structures. In one case, the response also reached the max_tokens limit.

ii For a loan decision-support system, a low temperature (approximately 0.0–0.3) would be more suitable because the system needs to produce consistent and predictable decisions with minimal randomness.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
SUMMARY_PROMPT_V1 = "Summarize this:"

#   Run it on L002 and L006. Read the output critically.
for letter_id in ["L002", "L006"]:
    response = ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    )
    print(f"\n--- {letter_id} ---")
    print(response.choices[0].message.content)


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications factually and neutrally.
Do not invent or assume any details.
Keep the summary to 3-4 sentences."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
for letter_id in ["L002", "L006"]:
    response = ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )
    print(f"\n--- {letter_id} ---")
    print(response.choices[0].message.content)
    # TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for letter_id in ["L002", "L006"]:
    v1 = ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    )
    
    v2 = ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )

    print(f"\n========== {letter_id} ==========")
    print("\nV1:")
    print(v1.choices[0].message.content)
    
    print("\nV2:")
    print(v2.choices[0].message.content)


--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business but is optimistic it will improve after the festive season and is willing to repay the loan when he can, despite not having collateral.

--- L006 ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.

--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but exp

## Student Reasoning — Summarization Prompts
1. Issues with V1 and how V2 improved them

V1 sometimes included information that was not actually provided. For example, in L006, V1 stated that Kofi had “no prior experience,” even though the letter only mentioned that he had not yet started the businesses. V1 also described Kofi’s trustworthiness as “assurance,” which was not directly mentioned in the original text. V2 improved this by staying factual and neutral, avoiding unsupported assumptions, and following the required 3–4 sentence limit.

2. Why is “no invented details” important?

Loan decisions should rely only on accurate information provided by the applicant. Adding information that is not true or supported could unfairly influence the outcome of someone's loan application.

This type of error is known as hallucination in large language models (LLMs), where the model generates information that is unsupported or incorrect.

In [6]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_SYSTEM_PROMPT = """You extract structured information from loan applications.

Return ONLY a valid JSON object with EXACTLY these keys:
{
  "applicant_name": "string",
  "amount_ghs": number,
  "purpose": "string",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}

Rules:
- If a field is not stated in the letter, use null.
- Do not guess or invent information.
- amount_ghs, monthly_profit_ghs, and repayment_months must be numbers.
- has_collateral_or_guarantor must be true or false.
- Return ONLY the JSON object. No explanation or markdown.

Example:

Letter:
"Dear Loan Officer, my name is Ama Mensah. I am requesting GHS 8,000
to purchase equipment for my bakery. My current monthly profit is
GHS 1,500. I have a guarantor who will support the loan. I would like
to repay the loan over 12 months."

Output:
{
    "applicant_name": "Ama Mensah",
    "amount_ghs": 8000,
    "purpose": "purchase equipment for my bakery",
    "monthly_profit_ghs": 1500,
    "has_collateral_or_guarantor": true,
    "repayment_months": 12
}
"""

EXTRACT_PROMPT = """Extract the required information from this loan application.

Loan application:
{letter_text}
"""

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    response = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=0.0
    )

    result = response.choices[0].message.content.strip()

    # Remove markdown JSON fences if the model adds them
    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    result = result.removesuffix("```")

    result = result.strip()

    try:
        data = json.loads(result)
        return data
    except json.JSONDecodeError:
        print("Warning: Could not parse LLM response as JSON.")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []
for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)

df = pd.DataFrame(results)

# Put letter_id first
columns = ["letter_id"] + [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

df = df[columns]

display(df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers for my poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


## Student Reasoning — Structured Extraction

**1. Why should the few-shot example be different?**
The example should not be taken from the six letters because the model might simply memorize or copy information from the data being tested. Using a separate example helps determine whether the prompt can work effectively with new and unseen information.

**2. Why use null, do not guess?**
This instruction helps stop the model from creating information that is missing from the letter. Without it, the model might try to assume, estimate, or invent values that were never provided.

**3. Why use temperature = 0?**
A temperature of 0 is suitable for extraction because it produces more consistent and predictable results. In contrast, creative tasks can benefit from a higher temperature because it allows the model to generate more diverse and creative responses.

In [7]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer.

Your task is to analyze a loan application and provide a factual decision-support brief.

Your response must contain exactly these four sections:

1. Strengths
- Bullet points grounded only in the letter.

2. Risks / red flags
- Bullet points based only on information in the letter.

3. Missing information
- List important information the loan officer should request.

4. Suggested next step
- Suggest an action such as "invite for interview", "request documents",
  or "flag for senior review".
- Do NOT recommend approving or rejecting the loan.

Important:
- Do not invent or assume information.
- Clearly distinguish stated facts from missing information.
- Final loan decisions are made by human loan officers, not by the AI.
"""

BRIEF_PROMPT = """Analyze this loan application using the extracted information below.

Loan application:
{letter_text}

Extracted information:
{extracted_json}
"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
briefs = {}

for letter_id, letter_text in LETTERS.items():
    row = df[df["letter_id"] == letter_id]

    if row.empty:
        briefs[letter_id] = "Extraction failed; no brief generated."
        continue

    extracted_json = row.iloc[0].to_dict()

    response = ask_llm(
        BRIEF_PROMPT.format(
            letter_text=letter_text,
            extracted_json=json.dumps(extracted_json, indent=2)
        ),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0.0
    )

    briefs[letter_id] = response.choices[0].message.content
# Print the briefs for L001, L002, and L006
for letter_id in ["L001", "L002", "L006"]:
    print()
    print(f"==== BRIEF — {letter_id} ====")
    print()
    print(briefs[letter_id])


==== BRIEF — L001 ====

## Step 1: Strengths
- The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
- She has a regular profit of GHS 900 each month from her current stall, showing a consistent income stream.
- Akosua has saved GHS 2,500 over two years with the susu scheme and has never missed a contribution, demonstrating her ability to save and commit to financial obligations.
- She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of financial security.

## Step 2: Risks / red flags
- The applicant is seeking a significant loan of GHS 8,000, which is a large amount compared to her monthly profit and savings, potentially straining her financial capabilities.
- Expanding into frozen foods with a deep freezer may require additional expenses and could pose operational risks if not managed properly.
- The repayment plan of GHS 450 monthly over 20 months may be ambitious 

## Comparison of L003 and L006

L003 appears to be the stronger loan application. Efua Darko has an established and registered dressmaking business, employs three apprentices, and has 18 months of sales records. She also reports an average monthly profit of GHS 2,800 and has a GHS 5,000 fixed deposit that could be used as security. These details provide clear evidence supporting her application. However, the main concerns are that the GHS 15,000 requested is intended for business expansion and that the officer should verify the sales records, business performance, and fixed deposit before proceeding.

L006 is considerably weaker based on the available information. Kofi has not yet started any of the three businesses he proposed, has no collateral, and does not provide evidence of existing business income or profits. He also plans to repay the GHS 50,000 loan within one year after the businesses become successful. This creates a significant risk because repayment relies on businesses that have not yet been established. Therefore, the system should request additional information, such as a detailed business plan, projected cash flows, estimated startup costs, and proof of available capital.

## Why should “approve” and “reject” be prohibited?

The AI should focus on identifying relevant evidence, strengths, risks, and missing information rather than making the final loan decision. The human loan officer should review this information and make the final approval or rejection decision.

Commit hash: 844c0df8c115d997b1d958acfda1ea0782e962b6

In [9]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).


# Compare extracted values with GOLD values field by field.
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

rows = []

for field in fields:
    row = {"field": field}
    correct = 0

    for letter_id in GOLD:
        predicted = df.loc[df["letter_id"] == letter_id, field].iloc[0]
        expected = GOLD[letter_id][field]

        # Name comparison is case-insensitive
        if field == "applicant_name":
            match = str(predicted).strip().lower() == str(expected).strip().lower()
        else:
            match = predicted == expected

        row[letter_id] = "✓" if match else "✗"

        if match:
            correct += 1

    row["accuracy"] = f"{correct}/3 ({correct / 3:.0%})"
    rows.append(row)

accuracy_df = pd.DataFrame(rows)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
display(accuracy_df)

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,3/3 (100%)
1,amount_ghs,✓,✓,✓,3/3 (100%)
2,purpose,✗,✗,✗,0/3 (0%)
3,monthly_profit_ghs,✓,✓,✗,2/3 (67%)
4,has_collateral_or_guarantor,✓,✓,✓,3/3 (100%)
5,repayment_months,✓,✓,✓,3/3 (100%)
